## EPA GHGRP 2023 — Facility-Level CO₂ Emissions Prediction

### Task Definition
The goal of this project is to **predict annual CO₂ emissions (metric tons, non-biogenic) at the individual facility level** using the EPA Greenhouse Gas Reporting Program (GHGRP) 2023 dataset. Each row in the dataset represents a single reporting facility for the 2023 calendar year.

This is a **supervised regression problem** on structured tabular data. Given facility-level attributes — sector, location, fuel and combustion mix, and individual GHG component emissions — we train models to estimate the facility's total CO₂ output for the year.

**Pipeline stages:**
1. Linear Regression               — interpretable baseline
2. 1-Hidden-Layer ReLU MLP         — minimal neural network baseline
3. Deep MLP **without** BatchNorm  — deep network, Dropout only
4. Deep MLP **with** BatchNorm     — deep network, BN + Dropout

**Dataset:** EPA GHGRP 2023, sheet "Direct Point Emitters"  
**Target:** `CO2 emissions (non-biogenic)` — metric tons CO₂ per facility per year  

**Key config toggle:**  
Set `SECTOR_FILTER = "Power Plants"` to train on power plants only (≈1,290 rows),  
or `SECTOR_FILTER = None` to train on all facilities (≈6,470 rows).

**Author:** Michael Paul

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
GHGRP_PATH  = "/content/EPA GHGRP 2023.xlsx"   # update path if needed
SHEET       = "Direct Point Emitters"
HEADER_ROW  = 3                                 # GHGRP data starts on row 4 (0-indexed: 3)
TARGET      = "CO2 emissions (non-biogenic) "   # metric tons CO2 — note trailing space in col name
TARGET_MAX  = 20_000_000                        # cap extreme outliers (max observed ~16.5M)

# ── TOGGLE THIS to switch between experiments ──────────────────────────────────
# Options:
#   SECTOR_FILTER = "Power Plants"   → train on power plants only  (~1,290 rows)
#   SECTOR_FILTER = None             → train on all facilities      (~6,470 rows)
SECTOR_FILTER = None
# ──────────────────────────────────────────────────────────────────────────────

RANDOM_SEED  = 42
TEST_SIZE    = 0.20
VAL_SIZE     = 0.10
BATCH_SIZE   = 256
EPOCHS_SMALL = 50
EPOCHS_DEEP  = 100
LR           = 1e-3

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

experiment_label = SECTOR_FILTER if SECTOR_FILTER else "All Sectors"
print(f"Experiment : {experiment_label}")

In [ ]:
# ── 1. Load & Inspect Data ─────────────────────────────────────────────────────
print("=" * 65)
print(f"STEP 1 — Load GHGRP 2023 | filter: {experiment_label}")
print("=" * 65)

df_raw = pd.read_excel(GHGRP_PATH, sheet_name=SHEET, header=HEADER_ROW)
print(f"Raw shape   : {df_raw.shape}")
print(f"\nSector distribution:")
print(df_raw["Industry Type (sectors)"].value_counts().head(10).to_string())

In [ ]:
# ── 1b. Missing Data Analysis ──────────────────────────────────────────────────
print("=" * 65)
print("STEP 1b — Missing Data Analysis")
print("=" * 65)

# Key feature columns we plan to use (flat list for inspection)
# Note: Latitude and Longitude removed from the feature set.
inspect_cols = [
    "State", "Industry Type (subparts)", "Industry Type (sectors)",
    "Primary NAICS Code",
    "Methane (CH4) emissions ", "Nitrous Oxide (N2O) emissions ",
    "HFC emissions", "PFC emissions", "SF6 emissions ",
    "Other GHGs (metric tons CO2e)", "Biogenic CO2 emissions (metric tons)",
    "Stationary Combustion", "Electricity Generation",
    "Ammonia Manufacturing", "Cement Production", "Hydrogen Production",
    "Iron and Steel Production", "Lime Production", "Nitric Acid Production",
    "Petroleum Refining", "Pulp and Paper Manufacturing",
    "Municipal Landfills", "Industrial Wastewater Treatment",
    "Industrial Waste Landfills",
    "Petroleum and Natural Gas Systems \u2013 Processing",
    "Petroleum and Natural Gas Systems \u2013 Transmission/Compression",
    "CO2 emissions (non-biogenic) "
]

# Only report columns that actually exist in the raw file
existing_inspect = [c for c in inspect_cols if c in df_raw.columns]
missing_summary = df_raw[existing_inspect].isnull().sum().reset_index()
missing_summary.columns = ["Column", "Missing Count"]
missing_summary["Missing %"] = (missing_summary["Missing Count"] / len(df_raw) * 100).round(2)
missing_summary = missing_summary[missing_summary["Missing Count"] > 0].sort_values(
    "Missing %", ascending=False
)

if len(missing_summary) == 0:
    print("No missing values in the selected feature columns.")
else:
    print(f"Columns with missing values ({len(missing_summary)} of {len(existing_inspect)}):\n")
    print(missing_summary.to_string(index=False))

print(f"\nImputation strategy:")
print("  Categorical cols → fill with 'UNKNOWN' before OneHotEncoder")
print("  GHG mix cols     → fill with column median (robust to outliers)")
print("  Process cols     → fill with 0 (not reported = not applicable)")
print("  Target           → rows with missing/out-of-range CO2 are dropped entirely")

In [ ]:
# ── 2. Sector Filter ───────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 2 — Apply sector filter")
print("=" * 65)

if SECTOR_FILTER is not None:
    df_filtered = df_raw[
        df_raw["Industry Type (sectors)"] == SECTOR_FILTER
    ].copy()
    print(f"Filtered to '{SECTOR_FILTER}': {len(df_filtered)} rows")
else:
    df_filtered = df_raw.copy()
    print(f"No filter applied — using all {len(df_filtered)} facilities")

# Drop rows missing the target
df_filtered[TARGET] = pd.to_numeric(df_filtered[TARGET], errors="coerce")
before = len(df_filtered)
df_filtered = df_filtered[
    df_filtered[TARGET].between(1, TARGET_MAX)
]
print(f"After target filter (1 – {TARGET_MAX:,} metric tons): "
      f"{before} → {len(df_filtered)} rows "
      f"({before - len(df_filtered)} removed)")

In [ ]:
# ── 3. Feature Selection ───────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 3 — Feature selection")
print("=" * 65)

# ── Categorical features — encoded with OneHotEncoder
# All are NOMINAL (unordered) categories; OHE is the correct choice.
#   State                    : 50+ unordered geographic labels
#   Industry Type (subparts) : unordered sector/process labels
#   NAICS_sector             : 2-digit NAICS prefix — an identifier, not a
#                              numeric quantity

# Derive 2-digit NAICS sector code from the full NAICS code
df_filtered["NAICS_sector"] = (
    df_filtered["Primary NAICS Code"]
    .astype(str).str[:2].str.strip()
)

cat_features = [
    "State",
    "Industry Type (subparts)",
    "NAICS_sector",
]
if SECTOR_FILTER is None:
    cat_features.append("Industry Type (sectors)")

# ── Individual GHG component columns
ghg_features = [
    #"Methane (CH4) emissions ",
    #"Nitrous Oxide (N2O) emissions ",
    #"HFC emissions",
    #"PFC emissions",
    #"SF6 emissions ",
    #"Other GHGs (metric tons CO2e)",
    #"Biogenic CO2 emissions (metric tons)",
]

# ── Process/subpart emission columns — sparse but informative
process_features = [
    "Stationary Combustion",
    "Electricity Generation",
    "Ammonia Manufacturing",
    "Cement Production",
    "Hydrogen Production",
    "Iron and Steel Production",
    "Lime Production",
    "Nitric Acid Production",
    "Petroleum Refining",
    "Pulp and Paper Manufacturing",
    #"Municipal Landfills",...Methane is main contributor
    #"Industrial Wastewater Treatment",...Methane is main contributor
    #"Industrial Waste Landfills",...Methane is main contributor
    #"Petroleum and Natural Gas Systems \u2013 Processing",...Methane is main contributor
    #"Petroleum and Natural Gas Systems \u2013 Transmission/Compression",...Methane is main contributor
]

all_features = cat_features + ghg_features + process_features

# Verify columns exist
existing = set(df_filtered.columns)
missing  = [c for c in all_features if c not in existing]
if missing:
    print(f"WARNING \u2014 columns not found: {missing}")
    all_features     = [c for c in all_features     if c in existing]
    cat_features     = [c for c in cat_features     if c in existing]
    ghg_features     = [c for c in ghg_features     if c in existing]
    process_features = [c for c in process_features if c in existing]
else:
    print("All feature columns verified OK.")

print(f"\nTotal features : {len(all_features)}")
print(f"  Categorical     : {len(cat_features)}  (OHE \u2014 nominal, unordered)")
print(f"  GHG mix         : {len(ghg_features)}  (median impute, passthrough)")
print(f"  Process/subpart : {len(process_features)}  (zero-fill, log1p, passthrough)")

In [ ]:
# ── 4. Preprocessing ───────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("STEP 4 \u2014 Preprocessing")
print("=" * 65)

df = df_filtered[all_features + [TARGET]].copy()

# ── Fill missing values ────────────────────────────────────────────────────────
for col in cat_features:
    df[col] = df[col].fillna("UNKNOWN").astype(str)

# GHG mix columns: median imputation (robust to extreme outliers)
for col in ghg_features:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())

# Process/subpart columns: zero-fill (not reported = facility does not run that process)
# then log1p to compress zero-heavy right skew
for col in process_features:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)
    df[col] = np.log1p(df[col].clip(lower=0))

# ── Target: log1p transform ────────────────────────────────────────────────────
y_raw = df[TARGET].values.astype(np.float32)
y     = np.log1p(y_raw)

print(f"Feature matrix (raw) : {df[all_features].shape}")
print(f"Target (raw)         : [{y_raw.min():,.0f}, {y_raw.max():,.0f}] metric tons CO2")
print(f"Target (log1p)       : [{y.min():.2f}, {y.max():.2f}]")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist(y_raw, bins=60, color="steelblue", edgecolor="white", linewidth=0.4)
axes[0].set_title(f"Raw CO2 \u2014 {experiment_label}")
axes[0].set_xlabel("Metric tons CO2")
axes[1].hist(y, bins=60, color="teal", edgecolor="white", linewidth=0.4)
axes[1].set_title(f"Log1p CO2 \u2014 {experiment_label}")
axes[1].set_xlabel("log1p(metric tons CO2)")
plt.tight_layout(); plt.show()

# ── Train / Validation / Test Split ───────────────────────────────────────────
# DATA LEAKAGE NOTE:
#   GHGRP 2023 is a cross-sectional snapshot: each row = one unique facility
#   for calendar year 2023. There is only ONE record per facility, so there is
#   no within-facility temporal or longitudinal overlap possible.
#   For multi-year GHGRP data (e.g. 2018-2023), a group-aware split by
#   facility ID would be required to prevent the same plant appearing in
#   both train and test.
X_raw = df[all_features].values

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
)
X_tr_raw, X_val_raw, y_tr, y_val = train_test_split(
    X_train_raw, y_train,
    test_size=VAL_SIZE / (1 - TEST_SIZE),
    random_state=RANDOM_SEED
)

# ── Build pandas DataFrames for ColumnTransformer ──────────────────────────────
def to_df(arr):
    return pd.DataFrame(arr, columns=all_features)

df_tr   = to_df(X_tr_raw)
df_val  = to_df(X_val_raw)
df_test = to_df(X_test_raw)

# ── Preprocessing pipeline ─────────────────────────────────────────────────────
# Encoder choice:
#   Categorical features (State, Industry Type, NAICS_sector) are NOMINAL.
#   OneHotEncoder is correct for nominal features
#
#   GHG mix and process features already received imputation and log1p above;
#   they are passed through without additional scaling.

preprocessor = ColumnTransformer(
    transformers=[
        ("ohe",  OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
        ("pass", "passthrough", ghg_features + process_features),
    ],
    remainder="drop"
)

X_tr_s   = preprocessor.fit_transform(df_tr).astype(np.float32)
X_val_s  = preprocessor.transform(df_val).astype(np.float32)
X_test_s = preprocessor.transform(df_test).astype(np.float32)

n_features = X_tr_s.shape[1]

# ── Track feature-group column ranges in the encoded output ───────────────────
# ColumnTransformer outputs columns in transformer order:
#   [OHE cols for categoricals] | [passthrough cols: ghg then process]
n_ohe_cols   = n_features - len(ghg_features) - len(process_features)
cat_col_idx  = list(range(0, n_ohe_cols))
ghg_col_idx  = list(range(n_ohe_cols, n_ohe_cols + len(ghg_features)))
proc_col_idx = list(range(n_ohe_cols + len(ghg_features), n_features))

print(f"\nFeature dimensions after encoding:")
print(f"  OHE categorical cols : {len(cat_col_idx)}")
print(f"  GHG mix cols         : {len(ghg_col_idx)}")
print(f"  Process cols         : {len(proc_col_idx)}")
print(f"  Total                : {n_features}")
print(f"\nTrain / Val / Test   : {len(X_tr_s)} / {len(X_val_s)} / {len(X_test_s)}")

In [ ]:
# ── 5. Evaluation Helper ───────────────────────────────────────────────────────
def evaluate(name, y_true_log, y_pred_log):
    """
    Evaluate on log1p scale (for R²) and back-transform
    to original metric tons for interpretable RMSE and MAE.
    """
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(np.clip(y_pred_log, 0, None))

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true_log, y_pred_log)

    print(f"\n{'─'*40}")
    print(f"  {name}")
    print(f"  RMSE : {rmse:>15,.0f} metric tons CO2")
    print(f"  MAE  : {mae:>15,.0f} metric tons CO2")
    print(f"  R²   : {r2:>15.4f}  (log scale)")
    return {"model": name, "RMSE": rmse, "MAE": mae, "R2": r2}

results = []

In [ ]:
# ── 6. MODEL 1 — Linear Regression ────────────────────────────────────────────
print("\n" + "=" * 65)
print(f"MODEL 1 \u2014 Linear Regression | {experiment_label}")
print("=" * 65)

lr_model  = LinearRegression()
lr_model.fit(X_tr_s, y_tr)
y_pred_lr = lr_model.predict(X_test_s)
results.append(evaluate("Linear Regression", y_test, y_pred_lr))

# Build full feature name list matching the ColumnTransformer output order
feature_names_out = (
    list(preprocessor.named_transformers_["ohe"].get_feature_names_out(cat_features))
    + ghg_features
    + process_features
)

coef_df = pd.DataFrame({
    "feature"    : feature_names_out,
    "coefficient": lr_model.coef_
})
coef_df = coef_df.reindex(
    coef_df["coefficient"].abs().sort_values(ascending=False).index
)
print("\nTop 10 coefficients (by |magnitude|):")
print(coef_df.head(10).to_string(index=False))

In [ ]:
# ── PyTorch helpers ────────────────────────────────────────────────────────────
def make_loader(X_np, y_np, shuffle=True):
    ds = TensorDataset(
        torch.from_numpy(X_np.astype(np.float32)),
        torch.from_numpy(y_np.astype(np.float32)).unsqueeze(1)
    )
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle)

def train_loop(model, optimizer, criterion, loader):
    model.train()
    total = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(xb)
    return total / len(loader.dataset)

def val_loop(model, criterion, loader):
    model.eval()
    total = 0.0
    with torch.no_grad():
        for xb, yb in loader:
            total += criterion(model(xb), yb).item() * len(xb)
    return total / len(loader.dataset)

def predict_np(model, X_np):
    model.eval()
    with torch.no_grad():
        return model(torch.from_numpy(X_np.astype(np.float32))).squeeze().numpy()

def run_training(model, train_loader, val_loader, epochs):
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()
    train_losses, val_losses = [], []
    for epoch in range(1, epochs + 1):
        tl = train_loop(model, optimizer, criterion, train_loader)
        vl = val_loop(model, criterion, val_loader)
        train_losses.append(tl)
        val_losses.append(vl)
        if epoch % max(1, epochs // 5) == 0:
            print(f"  Epoch {epoch:>4d}/{epochs}  "
                  f"train_RMSE={tl**0.5:.4f}  val_RMSE={vl**0.5:.4f} (log scale)")
    return train_losses, val_losses

train_loader = make_loader(X_tr_s, y_tr)
val_loader   = make_loader(X_val_s, y_val, shuffle=False)

In [ ]:
# ── 7. MODEL 2 — 1-Hidden-Layer ReLU MLP ──────────────────────────────────────
print("\n" + "=" * 65)
print(f"MODEL 2 — 1-Hidden-Layer ReLU MLP | {experiment_label}")
print("=" * 65)

class ShallowMLP(nn.Module):
    """Single hidden layer with ReLU activation."""
    def __init__(self, in_features, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )
    def forward(self, x):
        return self.net(x)

shallow_model = ShallowMLP(n_features, hidden=128)
print(shallow_model)
print(f"Parameters: {sum(p.numel() for p in shallow_model.parameters()):,}")

tl_s, vl_s = run_training(shallow_model, train_loader, val_loader, EPOCHS_SMALL)
y_pred_shallow = predict_np(shallow_model, X_test_s)

# Clip predictions to prevent numerical overflow during expm1 transformation
# The maximum log1p(CO2) in the dataset is ~16.61. Capping at 20.0 should be safe.
y_pred_shallow = np.clip(y_pred_shallow, 0, 20.0)

results.append(evaluate("1-Hidden-Layer ReLU MLP", y_test, y_pred_shallow))

In [ ]:
# ── 8. Deep MLP Architecture — with and without BatchNorm ─────────────────────
#
# The `use_bn` flag controls whether BatchNorm1d layers are inserted
# between each Linear → (BN) → ReLU → Dropout block.
#
# With BN    : Linear → BatchNorm1d → ReLU → Dropout
# Without BN : Linear →              ReLU → Dropout
#
# Both variants share the same hidden_dims and dropout rate so that
# any performance difference is attributable solely to BatchNorm.
#
# BatchNorm normalizes each mini-batch's activations to zero mean /
# unit variance, which can:
#   • accelerate convergence by reducing internal covariate shift
#   • act as a mild regularizer (reducing reliance on Dropout alone)
#   • smooth the loss landscape, allowing higher learning rates
# On small tabular datasets the benefit may be marginal or reversed;
# training both variants lets us measure the actual impact empirically.

class DeepMLP(nn.Module):
    """
    Configurable deep MLP with optional BatchNorm.

    Architecture per hidden layer:
        use_bn=True  → Linear → BatchNorm1d → ReLU → Dropout
        use_bn=False → Linear →              ReLU → Dropout
    """
    def __init__(
        self,
        in_features: int,
        hidden_dims: tuple = (256, 128, 64),
        dropout: float = 0.3,
        use_bn: bool = True,
    ):
        super().__init__()
        self.use_bn = use_bn
        layers = []
        in_dim = in_features
        for h in hidden_dims:
            layers.append(nn.Linear(in_dim, h))
            if use_bn:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
# ── 9. MODEL 3 — Deep MLP WITHOUT BatchNorm ────────────────────────────────────
print("\n" + "=" * 65)
print(f"MODEL 3 — Deep MLP (no BatchNorm) | {experiment_label}")
print("=" * 65)
print("Architecture: Linear → ReLU → Dropout  (×3 hidden layers)")

deep_no_bn = DeepMLP(
    n_features,
    hidden_dims=(256, 128, 64),
    dropout=0.3,
    use_bn=False,
)
print(deep_no_bn)
print(f"Parameters: {sum(p.numel() for p in deep_no_bn.parameters()):,}")

tl_no_bn, vl_no_bn = run_training(deep_no_bn, train_loader, val_loader, EPOCHS_DEEP)
y_pred_no_bn = np.clip(predict_np(deep_no_bn, X_test_s), 0, 20.0)
results.append(evaluate("Deep MLP (no BN)", y_test, y_pred_no_bn))

In [ ]:
# ── 10. MODEL 4 — Deep MLP WITH BatchNorm ──────────────────────────────────────
print("\n" + "=" * 65)
print(f"MODEL 4 — Deep MLP (with BatchNorm) | {experiment_label}")
print("=" * 65)
print("Architecture: Linear → BatchNorm1d → ReLU → Dropout  (×3 hidden layers)")

deep_bn = DeepMLP(
    n_features,
    hidden_dims=(256, 128, 64),
    dropout=0.3,
    use_bn=True,
)
print(deep_bn)
print(f"Parameters: {sum(p.numel() for p in deep_bn.parameters()):,}")

tl_bn, vl_bn = run_training(deep_bn, train_loader, val_loader, EPOCHS_DEEP)
y_pred_bn = np.clip(predict_np(deep_bn, X_test_s), 0, 20.0)
results.append(evaluate("Deep MLP (with BN)", y_test, y_pred_bn))

In [ ]:
# ── 11. Results Summary ─────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print(f"RESULTS SUMMARY — {experiment_label}")
print("=" * 65)
results_df = pd.DataFrame(results).set_index("model")
print(results_df.round(4).to_string())

# ── BatchNorm delta ────────────────────────────────────────────────────────────
bn_r2   = results_df.loc["Deep MLP (with BN)",  "R2"]
no_bn_r2= results_df.loc["Deep MLP (no BN)",    "R2"]
bn_rmse = results_df.loc["Deep MLP (with BN)",  "RMSE"]
no_bn_rmse = results_df.loc["Deep MLP (no BN)", "RMSE"]

print(f"\nBatchNorm impact (with BN vs no BN):")
print(f"  \u0394 R\u00b2   = {bn_r2 - no_bn_r2:+.4f}  (positive = BN helps)")
print(f"  \u0394 RMSE = {(bn_rmse - no_bn_rmse):+,.0f} metric tons  (negative = BN helps)")

In [ ]:
# ── 12. Plots ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle(
    f"CO\u2082 Emissions Prediction — Model Comparison ({experiment_label})",
    fontsize=14
)

# ── Panel 1: RMSE & MAE bar chart ─────────────────────────────────────────────
results_df[["RMSE", "MAE"]].plot(kind="bar", ax=axes[0, 0], rot=25, color=["steelblue", "coral"])
axes[0, 0].set_title("RMSE & MAE by Model (metric tons CO2)")
axes[0, 0].set_ylabel("Metric tons CO2")
axes[0, 0].tick_params(axis="x", labelsize=8)

# ── Panel 2: R² bar chart ─────────────────────────────────────────────────────
results_df["R2"].plot(kind="bar", ax=axes[0, 1], rot=25, color="mediumseagreen")
axes[0, 1].set_title("R\u00b2 by Model (log scale)")
axes[0, 1].set_ylabel("R\u00b2")
axes[0, 1].set_ylim(0, 1)
axes[0, 1].tick_params(axis="x", labelsize=8)

# ── Panel 3: Training curves — Deep MLP no BN ─────────────────────────────────
axes[1, 0].plot(np.sqrt(tl_no_bn), label="Train RMSE", linestyle="--", color="royalblue")
axes[1, 0].plot(np.sqrt(vl_no_bn), label="Val RMSE",               color="royalblue")
axes[1, 0].set_title("Deep MLP — No BatchNorm: Training Curves")
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("RMSE (log1p scale)")
axes[1, 0].legend(fontsize=9)

# ── Panel 4: Training curves — Deep MLP with BN ───────────────────────────────
axes[1, 1].plot(np.sqrt(tl_bn), label="Train RMSE", linestyle="--", color="darkorange")
axes[1, 1].plot(np.sqrt(vl_bn), label="Val RMSE",               color="darkorange")
axes[1, 1].set_title("Deep MLP — With BatchNorm: Training Curves")
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("RMSE (log1p scale)")
axes[1, 1].legend(fontsize=9)

plt.tight_layout()
fname = f"results_{experiment_label.replace(' ', '_').lower()}.png"
plt.savefig(fname, dpi=150)
plt.show()
print(f"\nPlot saved to {fname}")

In [ ]:
# ── 13. Predicted vs Actual — Deep MLP with BN vs no BN ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"Predicted vs Actual (log1p scale) — {experiment_label}",
    fontsize=13
)

for ax, y_pred, title, color in [
    (axes[0], y_pred_no_bn, "Deep MLP — No BatchNorm",   "royalblue"),
    (axes[1], y_pred_bn,    "Deep MLP — With BatchNorm", "darkorange"),
]:
    ax.scatter(y_test, y_pred, alpha=0.3, s=10, color=color)
    lim = [
        float(y_test.min()) - 0.5,
        float(max(y_test.max(), y_pred.max())) + 0.5,
    ]
    ax.plot(lim, lim, "r--", linewidth=1, label="Perfect fit")
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_title(title)
    ax.set_xlabel("Actual log1p(CO2 tons)")
    ax.set_ylabel("Predicted log1p(CO2 tons)")
    ax.legend(fontsize=8)

plt.tight_layout()
scatter_fname = f"scatter_{experiment_label.replace(' ', '_').lower()}.png"
plt.savefig(scatter_fname, dpi=150)
plt.show()
print(f"Scatter plot saved to {scatter_fname}")

In [ ]:
# ── 14. Ablation Study (Deep MLP with BN — baseline) ──────────────────────────
#   Ablation analysis measures each feature group's contribution by training
#   the same model architecture with that group *removed* from the input.
#   A larger degradation in R2 or RMSE indicates greater importance.
#
#     1. No Categorical Features  (State, Industry Type, NAICS_sector)
#     2. No GHG Mix Features      (CH4, N2O, HFCs, biogenic CO2, etc.) #GHG Mix is removed
#     3. No Process Features      (Stationary Combustion, Elec Gen, etc.)

print("\n" + "=" * 65)
print(f"ABLATION STUDY \u2014 Deep MLP (with BN) | {experiment_label}")
print("=" * 65)
print("Baseline: Deep MLP with BatchNorm trained above.")
print(f"Baseline RMSE = {results_df.loc['Deep MLP (with BN)', 'RMSE']:,.0f}")
print(f"Baseline R\u00b2   = {results_df.loc['Deep MLP (with BN)', 'R2']:.4f}")

baseline_rmse = results_df.loc["Deep MLP (with BN)", "RMSE"]
baseline_r2   = results_df.loc["Deep MLP (with BN)", "R2"]

# Column index ranges computed in Cell 4 (OHE-aware, no continuous cols)
ablation_groups = {
    "No Categorical Features": cat_col_idx,
    #"No GHG Mix Features":     ghg_col_idx,
    "No Process Features":     proc_col_idx,
}

ablation_results = []
all_col_idx = set(range(n_features))

for abl_name, removed_idx in ablation_groups.items():
    print(f"\n--- Ablating: {abl_name} ---")
    kept = sorted(all_col_idx - set(removed_idx))
    X_tr_abl   = X_tr_s[:, kept]
    X_val_abl  = X_val_s[:, kept]
    X_test_abl = X_test_s[:, kept]
    print(f"  Features remaining: {len(kept)}")

    model_abl = DeepMLP(len(kept), hidden_dims=(256, 128, 64), dropout=0.3, use_bn=True)
    tl_abl = make_loader(X_tr_abl, y_tr)
    vl_abl = make_loader(X_val_abl, y_val, shuffle=False)
    run_training(model_abl, tl_abl, vl_abl, EPOCHS_DEEP)

    y_pred_abl = predict_np(model_abl, X_test_abl)
    res = evaluate(abl_name, y_test, y_pred_abl)
    res["R2_Impact"]       = baseline_r2   - res["R2"]
    res["RMSE_Impact_Pct"] = (res["RMSE"] - baseline_rmse) / baseline_rmse * 100
    ablation_results.append(res)

ablation_df = pd.DataFrame(ablation_results).set_index("model")

print("\n" + "=" * 65)
print("Ablation Analysis Summary")
print("=" * 65)
print(ablation_df[["RMSE", "MAE", "R2", "R2_Impact", "RMSE_Impact_Pct"]].round(4).to_string())

most_r2   = ablation_df["R2_Impact"].idxmax()
most_rmse = ablation_df["RMSE_Impact_Pct"].idxmax()
print(f"\nGreatest R\u00b2 drop   : '{most_r2}'  (\u0394 R\u00b2 = {ablation_df.loc[most_r2, 'R2_Impact']:.4f})")
print(f"Greatest RMSE rise : '{most_rmse}'  (\u0394 RMSE = {ablation_df.loc[most_rmse, 'RMSE_Impact_Pct']:.2f}%)")

# Final results summary
print("\n" + "=" * 65)
print(f"FINAL RESULTS SUMMARY \u2014 {experiment_label}")
print("=" * 65)
print(pd.DataFrame(results).set_index("model").round(4).to_string())